# 🚀 Video2Video – Lanzador Optimizado para Kaggle (GPU Dual T4 ×2)

### ⚙️ Configuración previa en la barra lateral derecha de Kaggle:
1. **Accelerator**: Selecciona **GPU T4 x2** (te otorga 32 GB VRAM combinada).
2. **Internet**: Asegúrate de que esté en **Internet on** (requiere verificación telefónica gratuita de Kaggle).
3. **Secrets**: En `Add-ons` -> `Secrets`, agrega tus claves:
   - `GHP_GITHUB_TOKEN` *(opcional si el repo es público, obligatorio si es privado)*
   - `GOOGLE_API_KEY` *(para Gemini Flash / traducción y títulos virales)*
   - `FISH_API_KEY` *(para síntesis de voz Fish Audio TTS)*

   - `HF_TOKEN` *(obligatorio para Pyannote Diarization 3.1)*

In [ ]:
# @title ⚡ Arrancar V2V + OpenReel (2x T4 + Cloudflare Tunnel)
import os, time, threading, sys, warnings, subprocess, re, urllib.request
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

print("=" * 60)
print("🎬 INICIANDO VIDEO2VIDEO EN KAGGLE (DUAL T4 ACCELERATOR)")
print("=" * 60)

# --- 1. DETECCIÓN DE HARDWARE MULTI-GPU ---
try:
    import torch
    gpu_count = torch.cuda.device_count()
    print(f"🔥 GPUs NVIDIA detectadas: {gpu_count}")
    for i in range(gpu_count):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")
    
    if gpu_count >= 2:
        os.environ["V2V_DEVICE"] = "cuda:0"
        os.environ["V2V_DEVICE_SECONDARY"] = "cuda:1"
        print("⚡ Modo Dual-GPU activado: GPU 0 (Whisper+Diarización) | GPU 1 (RoFormer+Render)")
    elif gpu_count == 1:
        os.environ["V2V_DEVICE"] = "cuda:0"
        os.environ["V2V_DEVICE_SECONDARY"] = "cuda:0"
        print("⚡ Modo Single-GPU activado: GPU 0")
    else:
        print("⚠️ No se detectó GPU. Recuerda activar 'GPU T4 x2' en la barra lateral de Kaggle.")
except Exception as e:
    print("⚠️ Error comprobando GPUs:", e)

# --- 2. CONFIGURACIÓN Y SECRETOS DE KAGGLE ---
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_name in ["GHP_GITHUB_TOKEN", "FISH_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"]:
        try:
            val = user_secrets.get_secret(secret_name)
            if val:
                os.environ[secret_name] = val
                # Mostrar solo últimos 4 caracteres por seguridad
                masked = f"...{val[-4:]}" if len(val) >= 4 else "***"
                print(f"🔑 Secreto cargado: {secret_name} ({masked})")
            else:
                print(f"⚠️ Secreto '{secret_name}' está vacío en Kaggle.")
        except Exception as err:
            print(f"⚠️ No se pudo leer el secreto '{secret_name}'. Recuerda activar el interruptor de este secreto en Add-ons -> Secrets para este notebook.")
    if os.getenv("HF_TOKEN"):
        os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
        try:
            from huggingface_hub import login
            login(token=os.environ["HF_TOKEN"], write_permission=False)
            print("🤗 Hugging Face autenticado con éxito.")
        except Exception:
            pass
except Exception:
    print("ℹ️ Kaggle secrets no disponibles o no configurados en Add-ons.")

GHP_TOKEN = os.getenv("GHP_GITHUB_TOKEN", "")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git" if GHP_TOKEN else "https://github.com/almagomen/v2v.git"
REPO_PATH = Path("/kaggle/working/v2v")
BRANCH_NAME = "monorepo-subtree"
BACKEND_PORT, FRONTEND_PORT = 8000, 5173
LOG_FILE = REPO_PATH / "backend.log"

# --- 3. CLONADO Y ACTUALIZACIÓN INTELIGENTE ---
if not REPO_PATH.exists():
    print("📥 Clonando repositorio en /kaggle/working/v2v...")
    subprocess.run(["git", "clone", "-q", "-b", BRANCH_NAME, REPO_URL, str(REPO_PATH)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", "HEAD", "-q"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "origin", BRANCH_NAME, "-q"], check=True)
    diff = subprocess.run(["git", "-C", str(REPO_PATH), "diff", "--name-status", "HEAD", f"origin/{BRANCH_NAME}"],
                          capture_output=True, text=True).stdout.strip()
    if diff:
        print("🚀 ¡Cambios detectados en el repo! Sincronizando con upstream...")
        subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{BRANCH_NAME}", "-q"], check=True)
    else:
        print("✔️ Repositorio local al día.")

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# --- 4. INSTALACIÓN DE DEPENDENCIAS ---
if 'models_loaded' not in globals():
    print("📦 Instalando herramientas y paquetes optimizados con uv...")
    subprocess.run("pip install -q uv", shell=True)
    subprocess.run("curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1", shell=True)
    deno_path = "/root/.deno/bin"
    if deno_path not in os.environ["PATH"]:
        os.environ["PATH"] += f":{deno_path}"
    subprocess.run("npm install -g pnpm --silent >/dev/null 2>&1", shell=True)

    # Paquetes clave con --no-deps para velocidad máxima
    subprocess.run("uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q", shell=True)
    subprocess.run("uv pip install 'yt-dlp[default]' bs-roformer-infer tinydb sentencex soundfile 'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q", shell=True)
    subprocess.run("uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q", shell=True)
    subprocess.run("uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' -q", shell=True)

    # Descarga binario cloudflared para túnel HTTPS externo
    cf_bin = Path("/tmp/cloudflared")
    if not cf_bin.exists():
        print("🌐 Descargando Cloudflare Tunnel...")
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(cf_bin)], check=True)
        subprocess.run(["chmod", "+x", str(cf_bin)], check=True)

    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model

    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': get_roformer_model(),
        'models_loaded': True
    })
    print("✅ Modelos inicializados en memoria.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 5. SERVIDOR BACKEND UVICORN ---
import uvicorn
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    global server
    for m in [m for m in sys.modules if m.startswith(('api', 'src'))]:
        if 'models' in m: continue
        del sys.modules[m]
    
    from api.main import app
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_level="warning", loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config)
    server.run()

if 'server' in globals() and server.started:
    print("🛑 Reiniciando servidor backend...")
    server.should_exit = True
    time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# --- 6. FRONTEND (Vite / OpenReel) ---
OPENREEL_PATH = REPO_PATH / "openreel"
WEB_PATH = OPENREEL_PATH / "apps" / "web"
print("🏗️ Preparando dependencias de OpenReel...")
subprocess.run(["pnpm", "install", "--silent"], cwd=str(OPENREEL_PATH), check=True)
subprocess.run(["fuser", "-k", f"{FRONTEND_PORT}/tcp"], stderr=subprocess.DEVNULL)

f_env = os.environ.copy()
f_env["FORCE_COLOR"] = "1"
# Ejecutamos vite directamente desde el paquete apps/web con host 0.0.0.0
front_proc = subprocess.Popen(
    ["pnpm", "exec", "vite", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
    cwd=str(WEB_PATH), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=f_env
)

def log_reader(pipe):
    with open(LOG_FILE, "a") as f:
        for line in iter(pipe.readline, ''):
            if line.strip():
                f.write(f"[FRONTEND] {line.strip()}\n")
                f.flush()

threading.Thread(target=log_reader, args=(front_proc.stdout,), daemon=True).start()

# --- 7. VERIFICACIÓN DE SERVICIOS (HEALTHCHECKS) ---
print("⏳ Esperando que OpenReel (Vite) inicialice y responda en el puerto 5173...")
vite_ready = False
for i in range(45):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{FRONTEND_PORT}/")
        with urllib.request.urlopen(req, timeout=1) as resp:
            if resp.status in (200, 304):
                vite_ready = True
                print(f"✅ OpenReel Vite activo y respondiendo correctamente.")
                break
    except Exception:
        pass
    if front_proc.poll() is not None:
        print("❌ Error: El proceso de OpenReel terminó inesperadamente.")
        if LOG_FILE.exists():
            print("📋 Últimos logs del frontend:")
            print(LOG_FILE.read_text(encoding="utf-8")[-1000:])
        break
    time.sleep(1)

# --- 8. EXPOSICIÓN VÍA CLOUDFLARE TUNNEL ---
print("🌐 Levantando túnel HTTPS público para acceder a OpenReel...")
cf_proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{FRONTEND_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
for _ in range(40):
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

print("\n" + "=" * 60)
if tunnel_url:
    print("🎉 ¡ENTORNO LISTO Y FUNCIONANDO!")
    print("🔗 Abre OpenReel en tu navegador:")
    print(f"👉 {tunnel_url}")
    print("=" * 60)
    print("ℹ️ Esta celda se mantendrá en ejecución activa para sostener los servidores y el túnel.")
    print("   (Pulsa el botón de Stop ⏹️ cuando desees detener la sesión)\n")
    try:
        while True:
            time.sleep(2)
            if front_proc.poll() is not None:
                print("⚠️ El proceso de Vite se detuvo.")
                break
            if cf_proc.poll() is not None:
                print("⚠️ El proceso de Cloudflare Tunnel se detuvo.")
                break
    except KeyboardInterrupt:
        print("\n🛑 Servidores detenidos por el usuario.")
else:
    print("⚠️ No se pudo obtener la URL de Cloudflare automáticamente. Revisa los logs.")
print("=" * 60)


In [ ]:
# @title 📜 Ver últimos logs (Frontend y Backend)
import os
from pathlib import Path
log_path = Path('/kaggle/working/v2v/backend.log')
if log_path.exists():
    print(log_path.read_text(encoding='utf-8')[-2000:])
else:
    print('No hay archivo de log disponible aún.')
